# 手撕 QK Normalization

## 背景
对 Q 和 K 分别做 RMSNorm 再计算 attention，稳定训练。
QK-Norm 被 Qwen3、Gemma2、LLaMA-3.2 等广泛采用。
公式：Q' = Q / rms(Q), K' = K / rms(K), scores = Q' @ K'^T / sqrt(d)

## 考察点
- 为什么 QK-Norm 能稳定训练（控制 attention score 范围）
- RMSNorm vs LayerNorm（无 bias、无 mean centering）
- 与 softmax 温度的关系

In [ ]:
import torch
import torch.nn.functional as F
import math

def rms_norm(x, eps=1e-6):
    # x: (..., d)
    rms = x.pow(2).mean(dim=-1, keepdim=True).add(eps).rsqrt()
    return x * rms

def qk_norm_attention(q, k, v, scale=None):
    # q,k,v: (batch, n_heads, seq_len, d_head)
    d_head = q.size(-1)
    q_norm = rms_norm(q)
    k_norm = rms_norm(k)
    scale = scale or 1.0 / math.sqrt(d_head)
    scores = (q_norm @ k_norm.transpose(-2, -1)) * scale
    attn = F.softmax(scores, dim=-1)
    return attn @ v

In [ ]:
# 对比有/无 QK-Norm 的 attention score 范围
torch.manual_seed(42)
d_head = 128
q = torch.randn(1, 4, 32, d_head) * 10  # 大尺度 Q,K
k = torch.randn(1, 4, 32, d_head) * 10
v = torch.randn(1, 4, 32, d_head)
# 无 QK-Norm
scores_raw = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)
# 有 QK-Norm
q_n, k_n = rms_norm(q), rms_norm(k)
scores_norm = (q_n @ k_n.transpose(-2, -1)) / math.sqrt(d_head)
print(f"无 QK-Norm score 范围: [{scores_raw.min():.2f}, {scores_raw.max():.2f}]")
print(f"有 QK-Norm score 范围: [{scores_norm.min():.2f}, {scores_norm.max():.2f}]")
assert scores_norm.abs().max() < scores_raw.abs().max(), "QK-Norm 应缩小 score 范围"
out = qk_norm_attention(q, k, v)
assert out.shape == v.shape
print("✅ QK-Norm 有效缩小 attention score 范围，稳定训练")